The purpose of this notebook is to prepare the data for the Product Search solution accelerator.  You may find this notebook on https://github.com/databricks-industry-solutions/product-search.

##Introduction

In this notebook, we will access the [Wayfair Annotation Dataset (WANDS)](https://www.aboutwayfair.com/careers/tech-blog/wayfair-releases-wands-the-largest-and-richest-publicly-available-dataset-for-e-commerce-product-search-relevance), made accessible by [Wayfair](https://www.wayfair.com/) under an MIT License.

The dataset consists of three file types:
</p>

* Product - 42,000+ products features on the Wayfair website
* Query - 480 customer queries used for product searches
* Label - 233,000+ product results for the provided queries labeled for relevance

In the [Annotations Guidelines document](https://github.com/wayfair/WANDS/blob/main/Product%20Search%20Relevance%20Annotation%20Guidelines.pdf) that accompanies the dataset, Wayfair addresses the methods by which queries were labeled.  The three labels assigned to any query result are:
</p>

* Exact match - this label represents the surfaced product fully matches the search query
* Partial match - this label represents the surfaced product does not fully match the search query
* Irrelevant - this label indicates the product is not relevant to the query

As explained in the document, there is a bit of subjectivity in assigning these labels but the goal here is not to capture ground truth but instead to capture informed human judgement.

In [0]:
from pyspark.sql.types import *
import pyspark.sql.functions as fn

import os

In [0]:
%run "./00_Intro_and_Config"

##Step 1: Download Dataset Files

In this step, we will download the dataset files to a directory accessible within the Databricks workspace:

In [0]:
os.environ['WANDS_DOWNLOADS_PATH'] = '/dbfs'+ config['dbfs_path'] + '/downloads' 

In [0]:
%sh 

# delete any old copies of temp data
rm -rf $WANDS_DOWNLOADS_PATH

# make directory for temp tiles
mkdir -p $WANDS_DOWNLOADS_PATH

# move to temp directory
cd $WANDS_DOWNLOADS_PATH

# download datasets
wget -q https://raw.githubusercontent.com/wayfair/WANDS/main/dataset/label.csv
wget -q https://raw.githubusercontent.com/wayfair/WANDS/main/dataset/product.csv
wget -q https://raw.githubusercontent.com/wayfair/WANDS/main/dataset/query.csv

# show folder contents
pwd
ls -l

##Step 2: Write Data to Tables

In this step, we will read data from each of the previously downloaded files and write the data to tables that will make subsequent access easier and faster:

In [0]:
products_schema = StructType([
  StructField('product_id', IntegerType()),
  StructField('product_name', StringType()),
  StructField('product_class', StringType()),
  StructField('category_hierarchy', StringType()),
  StructField('product_description', StringType()),
  StructField('product_features', StringType()),
  StructField('rating_count', FloatType()),
  StructField('average_rating', FloatType()),
  StructField('review_count', FloatType())
  ])

_ = (
  spark
    .read
      .csv(
        path='dbfs:/wands/downloads/product.csv',
        sep='\t',
        header=True,
        schema=products_schema
        )
    .write
      .format('delta')
      .mode('overwrite')
      .option('overwriteSchema','true')
      .saveAsTable('products')
  )

display(
  spark.table('products')
  )

In [0]:
queries_schema = StructType([
  StructField('query_id', IntegerType()),
  StructField('query', StringType()),
  StructField('query_class', StringType())
  ])

_ = (
  spark
    .read
    .csv(
      path='dbfs:/wands/downloads/query.csv',
      sep='\t',
      header=True,
      schema=queries_schema
      )
    .write
      .format('delta')
      .mode('overwrite')
      .option('overwriteSchema','true')
      .saveAsTable('queries')
  )

display(
  spark.table('queries')
  )

In [0]:
labels_schema = StructType([
  StructField('id', IntegerType()),
  StructField('query_id', IntegerType()),
  StructField('product_id', IntegerType()),
  StructField('label', StringType())
  ])

_ = (
  spark
    .read
    .csv(
      path='dbfs:/wands/downloads/label.csv',
      sep='\t',
      header=True,
      schema=labels_schema
      )
    .write
      .format('delta')
      .mode('overwrite')
      .option('overwriteSchema','true')
      .saveAsTable('labels')
  )

display(spark.table('labels'))

##Step 3: Assign Label Scores

To prepare the text-based labels assigned to products returned by a query for use in our algorithm, we'll convert the labels to numerical scores based our judgement of how these labels should be weighted:

**NOTE** [This article](https://medium.com/@nikhilbd/how-to-measure-the-relevance-of-search-engines-18862479ebc) provides a nice discussion of how to approach the scoring of search results for relevance should you wish to explore alternative scoring patterns. 

In [0]:
if 'label_score' not in spark.table('labels').columns:
  _ = spark.sql('ALTER TABLE labels ADD COLUMN label_score FLOAT')

In [0]:
%sql

UPDATE labels
SET label_score = 
  CASE lower(label)
    WHEN 'exact' THEN 1.0
    WHEN 'partial' THEN 0.75
    WHEN 'irrelevant' THEN 0.0
    ELSE NULL
    END;

© 2023 Databricks, Inc. All rights reserved. The source in this notebook is provided subject to the Databricks License. All included or referenced third party libraries are subject to the licenses set forth below.

| library                                | description             | license    | source                                              |
|----------------------------------------|-------------------------|------------|-----------------------------------------------------|
|  WANDS | Wayfair product search relevance data | MIT  | https://github.com/wayfair/WANDS   |
| langchain | Building applications with LLMs through composability | MIT  |   https://pypi.org/project/langchain/ |
| chromadb | An open source embedding database |  Apache |  https://pypi.org/project/chromadb/  |
| sentence-transformers | Compute dense vector representations for sentences, paragraphs, and images | Apache 2.0 |https://pypi.org/project/sentence-transformers/ |